# Residual networks (skip connections)

_Deep Learning (CS230)_

**Add the layer's input back to its output, so very deep nets train instead of stalling.**

Stack enough plain layers and something surprising happens: the deeper network trains worse
       than a shallower one &mdash; worse training error, not just worse test error. This is the
       degradation problem. It is not overfitting (overfitting would still drive training error down);
       the deep net simply fails to optimize.

       That is strange, because a deep net could match a shallow one: make the extra layers do
       nothing (the identity function) and you recover the shallow net. The trouble is that learning to be the
       identity is hard for a stack of nonlinear layers.

---

This notebook is a practice scaffold for the **Residual networks (skip connections)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# A residual block: two conv layers, then ADD the input back (the skip / shortcut).
class BasicBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.relu  = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x                                 # the skip connection: a^[l]
        out = self.relu(self.bn1(self.conv1(x)))     # z^[l+1] -> a^[l+1]
        out = self.bn2(self.conv2(out))              # z^[l+2]
        out = self.relu(out + identity)              # a^[l+2] = g(a^[l] + z^[l+2])
        return out

# A tiny ResNet: stem -> stack of residual blocks -> global pool -> linear head.
class TinyResNet(nn.Module):
    def __init__(self, n_blocks=6, channels=16, n_classes=3):
        super().__init__()
        self.stem   = nn.Conv2d(3, channels, 3, padding=1, bias=False)
        self.blocks = nn.Sequential(*[BasicBlock(channels) for _ in range(n_blocks)])
        self.head   = nn.Linear(channels, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = x.mean(dim=(2, 3))                        # global average pool
        return self.head(x)

# Tiny synthetic "images": 3 classes, each a noisy constant-color 8x8 patch.
N, C, H, W, K = 240, 3, 8, 8, 3
y = torch.randint(0, K, (N,))
base = torch.tensor([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]])[y]
X = base[:, :, None, None] + 0.3 * torch.randn(N, C, H, W)

model = TinyResNet()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

model.train()
for step in range(60):
    opt.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    opt.step()
    if step % 10 == 0 or step == 59:
        acc = (model(X).argmax(1) == y).float().mean().item()
        print(f"step {step:2d}  loss {loss.item():.4f}  acc {acc:.3f}")

# step  0  loss 1.8582  acc 0.033
# step 10  loss 0.0324  acc 1.000
# step 20  loss 0.0103  acc 1.000
# step 30  loss 0.0038  acc 1.000
# step 40  loss 0.0019  acc 1.000
# step 50  loss 0.0012  acc 1.000
# step 59  loss 0.0010  acc 1.000

## Visualize the data & results

_As a gradient back-propagates through a 50-layer network, does it survive to the early layers WITH vs WITHOUT skip connections?_

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
L = 50              # depth: 50 layers
n = 64              # hidden width
gain = 0.15         # sub-unit per-layer gain -> plain net gradient vanishes

# Build per-layer Jacobian factors (network linearized around its operating point).
# Plain layer:    a_{l+1} = J_l @ a_l            -> backward multiplies by J^T (gain<1)
# Residual layer: a_{l+1} = a_l + J_l @ a_l      -> backward multiplies by (I + J^T): the +1
# Use the SAME J_l for both runs so the only difference is the skip connection.
Js = []
for _ in range(L):
    A = rng.standard_normal((n, n))
    A = A / np.linalg.norm(A, 2)     # spectral norm 1
    Js.append(gain * A)

g0 = rng.standard_normal(n); g0 /= np.linalg.norm(g0)   # unit-norm gradient at the output
plain, resid = [], []
gp, gr = g0.copy(), g0.copy()
for l in reversed(range(L)):          # back-prop from output (layer L) to input (layer 1)
    plain.append(np.linalg.norm(gp))
    resid.append(np.linalg.norm(gr))
    gp = Js[l].T @ gp                 # plain: chain-rule product of Jacobians
    gr = gr + Js[l].T @ gr            # residual: (I + J^T) keeps the "+1" highway
plain.reverse(); resid.reverse()      # index 0 -> layer 1 (input side)

depths = np.arange(1, L + 1)
print("Plain   :", [[int(d), round(float(p), 4)] for d, p in zip(depths, plain)])
print("Residual:", [[int(d), round(float(r), 4)] for d, r in zip(depths, resid)])
# Plain    -> norm decays to ~0.0 by depth ~46, spiking only at the output (depth 50)
# Residual -> norm stays in [0.98, 1.11] across all 50 layers

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A team makes their plain CNN deeper, from 20 to 56 layers, and the training error goes UP. They assume it is overfitting and add dropout. Why is that diagnosis wrong, and what should they do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Separate the two failure modes: overfitting raises the test gap while training error keeps falling; here the training error itself rose. — _Rising training error means the deeper net cannot even fit the data &mdash; an optimization failure, not a generalization one._
- Name it: this is the degradation problem &mdash; a deeper plain stack trains worse even though it could in principle copy the shallow net via identity layers. — _Plain nonlinear layers find the identity mapping hard to learn, so extra depth hurts optimization._
- Switch the blocks to residual blocks: add the input back, $a^{[l+2]} = g(a^{[l]} + z^{[l+2]})$, so each block need only learn the residual $F(x)=H(x)-x$. — _The skip hands the layers the identity for free, so depth can no longer make training worse, and gradients keep a "+1" highway back to early layers._

**Answer:** It is not overfitting &mdash; training error rose, so the deep net cannot even optimize: the degradation problem. Dropout fights overfitting, the wrong target. The fix is skip connections: turn the blocks into residual blocks so each learns only $F(x)=H(x)-x$ and the identity path keeps gradients flowing. ResNet (Residual Network) was built exactly for this.

</details>

**Problem 2.** In a residual block, the input $a^{[l]}$ has 64 channels but the block's second layer outputs $z^{[l+2]}$ with 128 channels. You try $a^{[l]} + z^{[l+2]}$ and it errors. What is wrong and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check the add: $a^{[l]} + z^{[l+2]}$ is element-wise, so both must have the same shape &mdash; 64 channels cannot add to 128. — _The shortcut and the main path have to match in channels (and spatial size) to be summed._
- Put a 1&times;1 convolution on the shortcut to project $a^{[l]}$ from 64 to 128 channels (and match stride if you also downsampled). — _A 1&times;1 conv is a cheap, learnable per-pixel linear map that reshapes the channel count without touching spatial structure._
- Now add the projected shortcut to $z^{[l+2]}$ and apply $g$. — _Shapes match, so the residual sum and final activation work._

**Answer:** The element-wise add needs matching shapes, and 64 &ne; 128. Replace the identity shortcut with a projection shortcut: a 1&times;1 convolution on the skip path that maps $a^{[l]}$ to 128 channels (and applies the same stride if you downsampled). Then $a^{[l+2]} = g(\text{proj}(a^{[l]}) + z^{[l+2]})$ adds cleanly.

</details>

**Problem 3.** Why does ResNet-50 use a bottleneck block (1&times;1 &rarr; 3&times;3 &rarr; 1&times;1) instead of two 3&times;3 convolutions, and how does the skip connection survive the channel squeeze?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the bottleneck: the first 1&times;1 conv reduces channels (e.g. 256 &rarr; 64), the 3&times;3 conv does the spatial work cheaply at 64 channels, and the last 1&times;1 conv restores channels (64 &rarr; 256). — _Doing the expensive 3&times;3 convolution at a low channel count cuts the FLOPs (Floating-Point Operations) a lot, so very deep nets stay affordable._
- Trace the skip: the shortcut carries the 256-channel input around the whole three-conv stack and is added to the 256-channel output. — _The block input and output both have 256 channels, so an identity shortcut matches &mdash; the squeeze to 64 happens only inside the main path._
- If the block also changes channels or downsamples at its boundary, put a 1&times;1 projection on the skip. — _Same shape-matching rule as any residual block._

**Answer:** The bottleneck (1&times;1 &rarr; 3&times;3 &rarr; 1&times;1) squeezes channels down for the costly 3&times;3 conv and expands them back, so deep ResNets like ResNet-50 spend far fewer FLOPs (Floating-Point Operations). The skip connection wraps the whole three-conv stack: input and output are both at the full channel count, so an identity (or 1&times;1 projection) shortcut matches and the residual sum still works.

</details>